# LabOne Q — Extra Lesson: Installing a Multi-Qubit System on a ZQCS

The previous lectures ran on Gen2 instruments (SHFQC, HDAWG, PQSC). This lesson is what the first day
with a **ZQCS**, Zurich Instruments' Gen4 control system, looks like — in the order it happens:

1. **The device setup, line by line** — a single-shelf ZQCS with two MWM1404 microwave modules driving a
   4 × 7 lattice of **fixed-frequency transmons**, entangled by **cross-resonance (CR)** gates: every
   logical signal written out with the Python device-setup API, one ZQCS port per line
2. **Line calibration** — LOs, IFs and ranges for every line, then *save it once, load it everywhere*
3. **A QPU on top, and the first three experiments** — `resonator_spectroscopy`, `amplitude_rabi` and
   `ramsey` from the Applications Library, on many qubits at once
4. **Every line at once, in the plain DSL** — an installation smoke test that fires every drive line, every
   CR line and every readout of the system, written in full LabOne Q DSL with no helper and no workflow,
   to show what the DSL looks like underneath everything the library builds

**This notebook needs the ZQCS device driver** (`zhinst-zqcs`, shipped with a ZQCS installation — not on
PyPI). On the open LabOne Q release the other lectures use, `session.connect` stops with
`Unknown device driver ZQCS`. Everything else — the setup API, the calibration, the DSL — is the same
LabOne Q. With the driver installed it runs in emulation (`use_emulation = True`) or on the rack.

In [ ]:
import numpy as np

from laboneq.simple import *
from laboneq import serializers
from laboneq.dsl.device.instruments import ZQCS
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

## 1. The device setup, line by line

A ZQCS is one instrument reached through its controller service — no data server. A port is
`shelf:slot:board:channel`. The system here is **one shelf with two MWM1404 modules**, in slots 2 and 11.
Each MWM1404 has two SG front ends (boards 1 and 4, channels 1–7) and two QA front ends (boards 2 and 3),
each with two QA channels: output 1 or 2, read back on input 3 or 4. That makes **28 drive channels and
8 QA channels**, and no flux channel: the qubits are fixed-frequency.

Until the chip mapping is known, the qubits are a placeholder **4 × 7 square lattice**: qubit `q{i}`
(row `i // 7`, column `i % 7`) takes drive channel `i` and QA channel `i % 8`, so **three or four qubits
share every readout line**, frequency-multiplexed. The 28 qubits use every drive and every QA channel of
the system; there is nothing spare.

Every logical signal is one `create_connection` line, with the port it is cabled to. This *is* the
cabling map: to rewire, edit the ports here.

In [ ]:
ZQCS_ADDRESS = "http://zqcs-controller.local"  # the controller service of your installation

setup = DeviceSetup(uid="lattice_4x7_installation")
setup.add_instruments(ZQCS(uid="zqcs", address=ZQCS_ADDRESS))

The 28 qubits, each with a drive, a measure (QA output) and an acquire (QA input) line. The lattice is a
checkerboard of *controls* (row + column even) and *targets*, for the CR gates below:

In [ ]:
# fmt: off
setup.add_connections(
    "zqcs",
    # q0: row 0, column 0 (control), readout line 0
    create_connection(to_signal="q0/drive",    ports="1:3:1:1", type="iq"),
    create_connection(to_signal="q0/measure",  ports="1:3:2:1", type="iq"),
    create_connection(to_signal="q0/acquire",  ports="1:3:2:3", type="acquire"),
    # q1: row 0, column 1 (target), readout line 1
    create_connection(to_signal="q1/drive",    ports="1:3:1:2", type="iq"),
    create_connection(to_signal="q1/measure",  ports="1:3:2:2", type="iq"),
    create_connection(to_signal="q1/acquire",  ports="1:3:2:4", type="acquire"),
    # q2: row 0, column 2 (control), readout line 2
    create_connection(to_signal="q2/drive",    ports="1:3:1:3", type="iq"),
    create_connection(to_signal="q2/measure",  ports="1:3:3:1", type="iq"),
    create_connection(to_signal="q2/acquire",  ports="1:3:3:3", type="acquire"),
    # q3: row 0, column 3 (target), readout line 3
    create_connection(to_signal="q3/drive",    ports="1:3:1:4", type="iq"),
    create_connection(to_signal="q3/measure",  ports="1:3:3:2", type="iq"),
    create_connection(to_signal="q3/acquire",  ports="1:3:3:4", type="acquire"),
    # q4: row 0, column 4 (control), readout line 4
    create_connection(to_signal="q4/drive",    ports="1:3:1:5", type="iq"),
    create_connection(to_signal="q4/measure",  ports="1:11:2:1", type="iq"),
    create_connection(to_signal="q4/acquire",  ports="1:11:2:3", type="acquire"),
    # q5: row 0, column 5 (target), readout line 5
    create_connection(to_signal="q5/drive",    ports="1:3:1:6", type="iq"),
    create_connection(to_signal="q5/measure",  ports="1:11:2:2", type="iq"),
    create_connection(to_signal="q5/acquire",  ports="1:11:2:4", type="acquire"),
    # q6: row 0, column 6 (control), readout line 6
    create_connection(to_signal="q6/drive",    ports="1:3:1:7", type="iq"),
    create_connection(to_signal="q6/measure",  ports="1:11:3:1", type="iq"),
    create_connection(to_signal="q6/acquire",  ports="1:11:3:3", type="acquire"),
    # q7: row 1, column 0 (target), readout line 7
    create_connection(to_signal="q7/drive",    ports="1:3:4:1", type="iq"),
    create_connection(to_signal="q7/measure",  ports="1:11:3:2", type="iq"),
    create_connection(to_signal="q7/acquire",  ports="1:11:3:4", type="acquire"),
    # q8: row 1, column 1 (control), readout line 0
    create_connection(to_signal="q8/drive",    ports="1:3:4:2", type="iq"),
    create_connection(to_signal="q8/measure",  ports="1:3:2:1", type="iq"),
    create_connection(to_signal="q8/acquire",  ports="1:3:2:3", type="acquire"),
    # q9: row 1, column 2 (target), readout line 1
    create_connection(to_signal="q9/drive",    ports="1:3:4:3", type="iq"),
    create_connection(to_signal="q9/measure",  ports="1:3:2:2", type="iq"),
    create_connection(to_signal="q9/acquire",  ports="1:3:2:4", type="acquire"),
    # q10: row 1, column 3 (control), readout line 2
    create_connection(to_signal="q10/drive",   ports="1:3:4:4", type="iq"),
    create_connection(to_signal="q10/measure", ports="1:3:3:1", type="iq"),
    create_connection(to_signal="q10/acquire", ports="1:3:3:3", type="acquire"),
    # q11: row 1, column 4 (target), readout line 3
    create_connection(to_signal="q11/drive",   ports="1:3:4:5", type="iq"),
    create_connection(to_signal="q11/measure", ports="1:3:3:2", type="iq"),
    create_connection(to_signal="q11/acquire", ports="1:3:3:4", type="acquire"),
    # q12: row 1, column 5 (control), readout line 4
    create_connection(to_signal="q12/drive",   ports="1:3:4:6", type="iq"),
    create_connection(to_signal="q12/measure", ports="1:11:2:1", type="iq"),
    create_connection(to_signal="q12/acquire", ports="1:11:2:3", type="acquire"),
    # q13: row 1, column 6 (target), readout line 5
    create_connection(to_signal="q13/drive",   ports="1:3:4:7", type="iq"),
    create_connection(to_signal="q13/measure", ports="1:11:2:2", type="iq"),
    create_connection(to_signal="q13/acquire", ports="1:11:2:4", type="acquire"),
    # q14: row 2, column 0 (control), readout line 6
    create_connection(to_signal="q14/drive",   ports="1:11:1:1", type="iq"),
    create_connection(to_signal="q14/measure", ports="1:11:3:1", type="iq"),
    create_connection(to_signal="q14/acquire", ports="1:11:3:3", type="acquire"),
    # q15: row 2, column 1 (target), readout line 7
    create_connection(to_signal="q15/drive",   ports="1:11:1:2", type="iq"),
    create_connection(to_signal="q15/measure", ports="1:11:3:2", type="iq"),
    create_connection(to_signal="q15/acquire", ports="1:11:3:4", type="acquire"),
    # q16: row 2, column 2 (control), readout line 0
    create_connection(to_signal="q16/drive",   ports="1:11:1:3", type="iq"),
    create_connection(to_signal="q16/measure", ports="1:3:2:1", type="iq"),
    create_connection(to_signal="q16/acquire", ports="1:3:2:3", type="acquire"),
    # q17: row 2, column 3 (target), readout line 1
    create_connection(to_signal="q17/drive",   ports="1:11:1:4", type="iq"),
    create_connection(to_signal="q17/measure", ports="1:3:2:2", type="iq"),
    create_connection(to_signal="q17/acquire", ports="1:3:2:4", type="acquire"),
    # q18: row 2, column 4 (control), readout line 2
    create_connection(to_signal="q18/drive",   ports="1:11:1:5", type="iq"),
    create_connection(to_signal="q18/measure", ports="1:3:3:1", type="iq"),
    create_connection(to_signal="q18/acquire", ports="1:3:3:3", type="acquire"),
    # q19: row 2, column 5 (target), readout line 3
    create_connection(to_signal="q19/drive",   ports="1:11:1:6", type="iq"),
    create_connection(to_signal="q19/measure", ports="1:3:3:2", type="iq"),
    create_connection(to_signal="q19/acquire", ports="1:3:3:4", type="acquire"),
    # q20: row 2, column 6 (control), readout line 4
    create_connection(to_signal="q20/drive",   ports="1:11:1:7", type="iq"),
    create_connection(to_signal="q20/measure", ports="1:11:2:1", type="iq"),
    create_connection(to_signal="q20/acquire", ports="1:11:2:3", type="acquire"),
    # q21: row 3, column 0 (target), readout line 5
    create_connection(to_signal="q21/drive",   ports="1:11:4:1", type="iq"),
    create_connection(to_signal="q21/measure", ports="1:11:2:2", type="iq"),
    create_connection(to_signal="q21/acquire", ports="1:11:2:4", type="acquire"),
    # q22: row 3, column 1 (control), readout line 6
    create_connection(to_signal="q22/drive",   ports="1:11:4:2", type="iq"),
    create_connection(to_signal="q22/measure", ports="1:11:3:1", type="iq"),
    create_connection(to_signal="q22/acquire", ports="1:11:3:3", type="acquire"),
    # q23: row 3, column 2 (target), readout line 7
    create_connection(to_signal="q23/drive",   ports="1:11:4:3", type="iq"),
    create_connection(to_signal="q23/measure", ports="1:11:3:2", type="iq"),
    create_connection(to_signal="q23/acquire", ports="1:11:3:4", type="acquire"),
    # q24: row 3, column 3 (control), readout line 0
    create_connection(to_signal="q24/drive",   ports="1:11:4:4", type="iq"),
    create_connection(to_signal="q24/measure", ports="1:3:2:1", type="iq"),
    create_connection(to_signal="q24/acquire", ports="1:3:2:3", type="acquire"),
    # q25: row 3, column 4 (target), readout line 1
    create_connection(to_signal="q25/drive",   ports="1:11:4:5", type="iq"),
    create_connection(to_signal="q25/measure", ports="1:3:2:2", type="iq"),
    create_connection(to_signal="q25/acquire", ports="1:3:2:4", type="acquire"),
    # q26: row 3, column 5 (control), readout line 2
    create_connection(to_signal="q26/drive",   ports="1:11:4:6", type="iq"),
    create_connection(to_signal="q26/measure", ports="1:3:3:1", type="iq"),
    create_connection(to_signal="q26/acquire", ports="1:3:3:3", type="acquire"),
    # q27: row 3, column 6 (target), readout line 3
    create_connection(to_signal="q27/drive",   ports="1:11:4:7", type="iq"),
    create_connection(to_signal="q27/measure", ports="1:3:3:2", type="iq"),
    create_connection(to_signal="q27/acquire", ports="1:3:3:4", type="acquire"),
)
# fmt: on

Fixed-frequency qubits have no coupler to tune: two neighbours are entangled by **cross resonance**,
driving the control qubit at the frequency of the target. That drive leaves through the control's own
port, but needs its own oscillator, so it is a logical signal of its own — a **CR line** — on the
control's drive port. One CR line per nearest-neighbour pair, from the control to the target: 45 of them,
up to four on one port (`q8` drives `q1`, `q7`, `q9` and `q15`):

In [ ]:
# fmt: off
setup.add_connections(
    "zqcs",
    create_connection(to_signal="cr_q0q1/drive",   ports="1:3:1:1", type="iq"),  # q0 -> q1
    create_connection(to_signal="cr_q0q7/drive",   ports="1:3:1:1", type="iq"),  # q0 -> q7
    create_connection(to_signal="cr_q2q1/drive",   ports="1:3:1:3", type="iq"),  # q2 -> q1
    create_connection(to_signal="cr_q8q1/drive",   ports="1:3:4:2", type="iq"),  # q8 -> q1
    create_connection(to_signal="cr_q2q3/drive",   ports="1:3:1:3", type="iq"),  # q2 -> q3
    create_connection(to_signal="cr_q2q9/drive",   ports="1:3:1:3", type="iq"),  # q2 -> q9
    create_connection(to_signal="cr_q4q3/drive",   ports="1:3:1:5", type="iq"),  # q4 -> q3
    create_connection(to_signal="cr_q10q3/drive",  ports="1:3:4:4", type="iq"),  # q10 -> q3
    create_connection(to_signal="cr_q4q5/drive",   ports="1:3:1:5", type="iq"),  # q4 -> q5
    create_connection(to_signal="cr_q4q11/drive",  ports="1:3:1:5", type="iq"),  # q4 -> q11
    create_connection(to_signal="cr_q6q5/drive",   ports="1:3:1:7", type="iq"),  # q6 -> q5
    create_connection(to_signal="cr_q12q5/drive",  ports="1:3:4:6", type="iq"),  # q12 -> q5
    create_connection(to_signal="cr_q6q13/drive",  ports="1:3:1:7", type="iq"),  # q6 -> q13
    create_connection(to_signal="cr_q8q7/drive",   ports="1:3:4:2", type="iq"),  # q8 -> q7
    create_connection(to_signal="cr_q14q7/drive",  ports="1:11:1:1", type="iq"),  # q14 -> q7
    create_connection(to_signal="cr_q8q9/drive",   ports="1:3:4:2", type="iq"),  # q8 -> q9
    create_connection(to_signal="cr_q8q15/drive",  ports="1:3:4:2", type="iq"),  # q8 -> q15
    create_connection(to_signal="cr_q10q9/drive",  ports="1:3:4:4", type="iq"),  # q10 -> q9
    create_connection(to_signal="cr_q16q9/drive",  ports="1:11:1:3", type="iq"),  # q16 -> q9
    create_connection(to_signal="cr_q10q11/drive", ports="1:3:4:4", type="iq"),  # q10 -> q11
    create_connection(to_signal="cr_q10q17/drive", ports="1:3:4:4", type="iq"),  # q10 -> q17
    create_connection(to_signal="cr_q12q11/drive", ports="1:3:4:6", type="iq"),  # q12 -> q11
    create_connection(to_signal="cr_q18q11/drive", ports="1:11:1:5", type="iq"),  # q18 -> q11
    create_connection(to_signal="cr_q12q13/drive", ports="1:3:4:6", type="iq"),  # q12 -> q13
    create_connection(to_signal="cr_q12q19/drive", ports="1:3:4:6", type="iq"),  # q12 -> q19
    create_connection(to_signal="cr_q20q13/drive", ports="1:11:1:7", type="iq"),  # q20 -> q13
    create_connection(to_signal="cr_q14q15/drive", ports="1:11:1:1", type="iq"),  # q14 -> q15
    create_connection(to_signal="cr_q14q21/drive", ports="1:11:1:1", type="iq"),  # q14 -> q21
    create_connection(to_signal="cr_q16q15/drive", ports="1:11:1:3", type="iq"),  # q16 -> q15
    create_connection(to_signal="cr_q22q15/drive", ports="1:11:4:2", type="iq"),  # q22 -> q15
    create_connection(to_signal="cr_q16q17/drive", ports="1:11:1:3", type="iq"),  # q16 -> q17
    create_connection(to_signal="cr_q16q23/drive", ports="1:11:1:3", type="iq"),  # q16 -> q23
    create_connection(to_signal="cr_q18q17/drive", ports="1:11:1:5", type="iq"),  # q18 -> q17
    create_connection(to_signal="cr_q24q17/drive", ports="1:11:4:4", type="iq"),  # q24 -> q17
    create_connection(to_signal="cr_q18q19/drive", ports="1:11:1:5", type="iq"),  # q18 -> q19
    create_connection(to_signal="cr_q18q25/drive", ports="1:11:1:5", type="iq"),  # q18 -> q25
    create_connection(to_signal="cr_q20q19/drive", ports="1:11:1:7", type="iq"),  # q20 -> q19
    create_connection(to_signal="cr_q26q19/drive", ports="1:11:4:6", type="iq"),  # q26 -> q19
    create_connection(to_signal="cr_q20q27/drive", ports="1:11:1:7", type="iq"),  # q20 -> q27
    create_connection(to_signal="cr_q22q21/drive", ports="1:11:4:2", type="iq"),  # q22 -> q21
    create_connection(to_signal="cr_q22q23/drive", ports="1:11:4:2", type="iq"),  # q22 -> q23
    create_connection(to_signal="cr_q24q23/drive", ports="1:11:4:4", type="iq"),  # q24 -> q23
    create_connection(to_signal="cr_q24q25/drive", ports="1:11:4:4", type="iq"),  # q24 -> q25
    create_connection(to_signal="cr_q26q25/drive", ports="1:11:4:6", type="iq"),  # q26 -> q25
    create_connection(to_signal="cr_q26q27/drive", ports="1:11:4:6", type="iq"),  # q26 -> q27
)
# fmt: on

In [ ]:
qubits = sorted((g for g in setup.logical_signal_groups if g.startswith("q")), key=lambda g: int(g[1:]))
cr_lines = [g for g in setup.logical_signal_groups if g.startswith("cr_")]
print(len(qubits), "qubits,", len(cr_lines), "CR lines")

# which qubits share a QA channel: group by the measure port
qa_channels = {}
for q in qubits:
    port = setup.logical_signal_groups[q].logical_signals["measure"].physical_channel.uid
    qa_channels.setdefault(port, []).append(q)
for port, channel_qubits in qa_channels.items():
    print(f"{port}: {', '.join(channel_qubits)}")

# which CR lines share a drive port with their control
drive_ports = {}
for line in cr_lines:
    port = setup.logical_signal_groups[line].logical_signals["drive"].physical_channel.uid
    drive_ports.setdefault(port, []).append(line)
print(f"{port}: {', '.join(drive_ports[port])}  (and the control's own drive)")

## 2. Line calibration

One setting per line type. The drive lines run on a HARDWARE oscillator at their maximum output range;
the readout lines on a SOFTWARE oscillator, with one LO per QA channel shared by measure (out) and acquire
(in), and the acquire input at its maximum range. The QA outputs have no attenuator on the ZQCS, so their
range stays unset — the compiler refuses one.

A **CR line** is a drive line on its control's port: the same LO and range as the control's drive (one
physical channel has one of each), its own hardware oscillator. On the ZQCS the signals of one channel
share one NCO, which switches between their frequencies phase-coherently.

The qubits on a readout line get IFs 60 MHz apart around 150 MHz — placeholders until the resonators are
measured, but already what a multiplexed line looks like: one LO per channel, one tone per qubit on a
**software** oscillator (a hardware oscillator is one per channel, so it cannot carry a frequency per
qubit).

In [ ]:
DRIVE_LO = 500e6
READOUT_LO = 8e9  # one LO per QA channel, the same on measure (out) and acquire (in)
DRIVE_RANGE_MAX = 14  # dBm
ACQUIRE_RANGE_MAX = 0  # dBm

readout_ifs = {}
for channel_qubits in qa_channels.values():
    for k, q in enumerate(channel_qubits):
        readout_ifs[q] = 150e6 + 60e6 * (k - (len(channel_qubits) - 1) / 2)

calibration = Calibration()
for q in qubits:
    calibration[f"/logical_signal_groups/{q}/drive"] = SignalCalibration(
        local_oscillator=Oscillator(frequency=DRIVE_LO),
        oscillator=Oscillator(frequency=100e6, modulation_type=ModulationType.HARDWARE),
        range=DRIVE_RANGE_MAX,
    )
    readout_oscillator = Oscillator(frequency=readout_ifs[q], modulation_type=ModulationType.SOFTWARE)
    calibration[f"/logical_signal_groups/{q}/measure"] = SignalCalibration(
        local_oscillator=Oscillator(frequency=READOUT_LO),
        oscillator=readout_oscillator,
    )
    calibration[f"/logical_signal_groups/{q}/acquire"] = SignalCalibration(
        local_oscillator=Oscillator(frequency=READOUT_LO),
        oscillator=readout_oscillator,
        range=ACQUIRE_RANGE_MAX,
    )
for line in cr_lines:
    # the LO and range of the control's channel, an oscillator of its own
    calibration[f"/logical_signal_groups/{line}/drive"] = SignalCalibration(
        local_oscillator=Oscillator(frequency=DRIVE_LO),
        oscillator=Oscillator(frequency=-100e6, modulation_type=ModulationType.HARDWARE),
        range=DRIVE_RANGE_MAX,
    )
setup.set_calibration(calibration)

### 2.1 Save it once, load it everywhere

Writing the setup out line by line and calibrating every line is installation work, done **once**. What it
produces is one file — every line, its port and its calibration — and from then on this is all a notebook
needs:

In [ ]:
serializers.save(setup, "lattice_4x7_setup.json")

setup = serializers.load("lattice_4x7_setup.json")

## 3. Session

In [ ]:
use_emulation = False  # False to run on the ZQCS

session = Session(device_setup=setup)
session.connect(do_emulation=use_emulation)

[2026.11.10 08:43:07.083] WARNING Connecting to the Slow Control at http://zqcs-controller.local/slow-control failed (attempt 1 of 2): Could not connect to http://zqcs-controller.local/slow-control after 6 attempts: Cannot connect to host zqcs-controller.local:80 ssl:default [Connect call failed ('zqcs-controller.local', 80)]


## 4. The QPU, and the first three experiments

`TunableTransmonQubit.from_device_setup` builds one qubit per logical signal group that follows the
`drive` / `measure` / `acquire` convention — the CR lines are left alone (they belong to two-qubit gates,
not to a qubit). The qubits here have no `flux` line: the qubit class treats it as optional, so a
fixed-frequency transmon is the same class with one line less. The parameters are the installation's first
guesses; every workflow below reads its pulses and frequencies from them, and in a real bring-up the
analyses write the measured values back (Lecture 6).

One thing the library's qubit does not know is which generation it runs on. Every experiment built with
`@dsl.qubit_experiment` applies `qubit.calibration()` as its experiment calibration, and for a Gen2 rack
that puts the acquisition delay on the acquire line as `port_delay` and a `range` on every line. On the
ZQCS `port_delay` is discontinued — its successor is `delay_signal` — the QA outputs take no range, and the
multiplexed readout needs a software oscillator. So the qubit for a Gen4 system is a small subclass: same
parameters, same operations, a calibration the hardware accepts.

Two more Gen4 rules show up in the parameters: every acquire line multiplexed on one QA input must carry
the **same** acquisition delay (one `readout_integration_delay` for all, on the 2 ns grid), and each shot
ends with a long wait — `reset_delay_length` of 400 µs — because the readout result buffer drains more
slowly than acquisitions fill it. That wait doubles as the passive qubit reset.

In [ ]:
from laboneq.dsl.quantum import QPU
from laboneq_applications.qpu_types.tunable_transmon import TunableTransmonQubit, TunableTransmonOperations
from laboneq_applications.experiments import resonator_spectroscopy, amplitude_rabi, ramsey

class Gen4TransmonQubit(TunableTransmonQubit):
    def calibration(self):
        calibration = super().calibration()
        acquire = calibration[self.signals["acquire"]]
        acquire.delay_signal, acquire.port_delay = acquire.port_delay, None
        calibration[self.signals["measure"]].range = None
        calibration[self.signals["measure"]].oscillator.modulation_type = ModulationType.SOFTWARE
        if "flux" in self.signals:  # none on a fixed-frequency qubit
            calibration[self.signals["flux"]].range = None
        return calibration


transmons = Gen4TransmonQubit.from_device_setup(setup, qubit_uids=qubits)
for i, q in enumerate(transmons):
    q.update(
        drive_lo_frequency=DRIVE_LO,
        resonance_frequency_ge=DRIVE_LO + 100e6 + 10e6 * i,
        readout_lo_frequency=READOUT_LO,
        readout_resonator_frequency=READOUT_LO + readout_ifs[q.uid],
        ge_drive_amplitude_pi=0.5,
        ge_drive_amplitude_pi2=0.25,
        ge_drive_length=32e-9,
        readout_length=1e-6,
        readout_integration_length=1e-6,
        readout_amplitude=0.1,
        readout_integration_delay=20e-9,  # the same on every qubit of a readout line
        reset_delay_length=400e-6,  # lets the readout result buffer drain between shots
        drive_range=DRIVE_RANGE_MAX,
        readout_range_in=ACQUIRE_RANGE_MAX,
    )
qpu = QPU(quantum_elements=transmons, quantum_operations=TunableTransmonOperations())
print(transmons[0].signals)

Resonator spectroscopy on the first qubit of each readout line, then an amplitude Rabi and a Ramsey on
all 28 qubits in parallel — three or four multiplexed tones per QA channel — with the same
`experiment_workflow` calls as Lecture 4 and lists of sweep points, one per qubit. Analysis is off: in
emulation there is no physics to fit, and on the rack the first spectroscopy is for looking, not fitting.

`reset_oscillator_phase(True)` on every workflow is the other Gen4 rule: the QA output does not reset the
phase of the software readout oscillator by itself, so without a reset at the start of each shot the
readout waveform depends on the absolute play time, and a delay sweep like the Ramsey bakes one waveform
per point and exhausts the waveform memory.

In [ ]:
print(opts)

In [ ]:
opts.find_peaks(True)
opts.fit_lorentzian(True)

In [ ]:
first_on_each_channel = [channel_qubits[0] for channel_qubits in qa_channels.values()]

opts = resonator_spectroscopy.experiment_workflow.options()
opts.count(4)
opts.do_analysis(True)
opts.close_figures(False)
opts.find_peaks(True)
opts.fit_lorentzian(True)

res_spec = resonator_spectroscopy.experiment_workflow(
    session, qpu, "q4",
    #frequencies=[np.linspace(READOUT_LO + 50e6, READOUT_LO + 350e6, 101)] * len(first_on_each_channel),
    frequencies=np.linspace(READOUT_LO +10e6, READOUT_LO + 200e6, 1001),
    options=opts,
).run()
print([t.name for t in res_spec.tasks])

In [ ]:
import matplotlib.pyplot as plt

plt.show()

In [ ]:
opts = amplitude_rabi.experiment_workflow.options()
opts.count(4)
opts.do_analysis(True)
opts.reset_oscillator_phase(True)

res_rabi = amplitude_rabi.experiment_workflow(
    session, qpu, qubits,
    amplitudes=[np.linspace(0, 1, 21)] * len(qubits),
    options=opts,
).run()

compiled_rabi = res_rabi.tasks["compile_experiment"].output
plot_simulation(compiled_rabi, length=2e-6, signal_names_to_show=["q0/drive", "q1/drive", "q0/measure", "q8/measure"])

In [ ]:
opts = ramsey.experiment_workflow.options()
opts.count(4)
opts.do_analysis(False)
opts.reset_oscillator_phase(True)

res_ramsey = ramsey.experiment_workflow(
    session, qpu, qubits,
    delays=[np.linspace(0, 2e-6, 21)] * len(qubits),
    detunings=[1e6] * len(qubits),
    options=opts,
).run()

show_pulse_sheet("ramsey_all_qubits", res_ramsey.tasks["compile_experiment"].output)

## 5. Every line at once — the plain DSL

An installation smoke test: drive on every qubit drive line, then every CR line, then the readout of every
qubit — every channel of the system plays in one shot, and every logical signal on it. No qubit object, no
quantum operation, no workflow: one `ExperimentSignal` per line, one `play` per line, and the sections that
order them.

The CR lines share their control's channel, and one channel plays one pulse at a time, so they go in
**layers**: layer `k` plays the `k`-th CR line of every control, all controls at once — four layers, as
`q8` has four. The qubits of a readout line play their readout pulses together on the one QA output, each
on its own IF, and the acquisitions run together on the one QA input — that is what the multiplexed
calibration above is for, and the phase reset before the readout is the manual form of the
`reset_oscillator_phase` option above. The 400 µs at the end is the result-buffer wait.

In [ ]:
drive_lines = [f"{q}/drive" for q in qubits]
cr_layers = []  # layer k: the k-th CR line of every control
for port_lines in drive_ports.values():
    for k, line in enumerate(port_lines):
        if k == len(cr_layers):
            cr_layers.append([])
        cr_layers[k].append(f"{line}/drive")
print(len(drive_lines), "drive lines,", sum(map(len, cr_layers)), "CR lines in", len(cr_layers), "layers,", len(qubits), "readouts")

drive_pulse = pulse_library.gaussian(uid="drive_pulse", length=800e-9, amplitude=0.9)
cr_pulse = pulse_library.gaussian_square(uid="cr_pulse", length=300e-9, amplitude=0.9)
readout_pulse = pulse_library.const(uid="readout_pulse", length=1e-6, amplitude=0.1)
integration_kernel = pulse_library.const(uid="integration_kernel", length=1e-6, amplitude=1.0)

all_lines = Experiment(
    uid="all_lines",
    signals=[ExperimentSignal(line, map_to=f"/logical_signal_groups/{line}") for line in drive_lines + sum(cr_layers, [])]
    + [ExperimentSignal(f"{q}/measure", map_to=f"/logical_signal_groups/{q}/measure") for q in qubits]
    + [ExperimentSignal(f"{q}/acquire", map_to=f"/logical_signal_groups/{q}/acquire") for q in qubits],
)

with all_lines.acquire_loop_rt(uid="shots", count=100, averaging_mode=AveragingMode.CYCLIC, acquisition_type=AcquisitionType.INTEGRATION):
    with all_lines.section(uid="drive"):
        for line in drive_lines:
            all_lines.play(signal=line, pulse=drive_pulse)
    previous = "drive"
    for k, layer in enumerate(cr_layers):
        with all_lines.section(uid=f"cr_layer_{k}", play_after=previous):
            for line in layer:
                all_lines.play(signal=line, pulse=cr_pulse)
        previous = f"cr_layer_{k}"
    with all_lines.section(uid="readout", play_after=previous):
        for q in qubits:
            all_lines.reset_oscillator_phase(signal=f"{q}/measure")
            all_lines.play(signal=f"{q}/measure", pulse=readout_pulse)
            all_lines.acquire(signal=f"{q}/acquire", handle=f"{q}/result", kernel=integration_kernel)
    with all_lines.section(uid="relax", play_after="readout"):
        for q in qubits:
            all_lines.delay(signal=f"{q}/measure", time=400e-6)

In [ ]:
compiled_all_lines = session.compile(all_lines)
results = session.run(compiled_all_lines)

print(len(all_lines.signals), "signals,", f"{compiled_all_lines.estimated_runtime * 1e3:.1f} ms per run")
print(sorted(results.acquired_results.keys()))

A few lines side by side: the drive port of `q8`, which plays its own drive and then its four CR lines one
layer after the other, the drive port of a target, and the four tones of one readout line — the same QA
output, four IFs.

In [ ]:
plot_simulation(
    compiled_all_lines,
    length=3e-6,
    signal_names_to_show=["q8/drive", "q1/drive"] + [f"q{i}/measure" for i in (0, 8, 16, 24)],
)

And the whole system in the pulse sheet:

In [ ]:
show_pulse_sheet("all_lines", compiled_all_lines)